In [1]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate

load_dotenv()  # Load environment variables from .env file

True

In [2]:
from langchain_groq import ChatGroq

def llm_instance(api_key: str, model_name: str = "llama-3.3-70b-versatile", temperature: float = 0.0) -> ChatGroq:
    """
    Set up the LLM with the provided API key and model name.

    Args:
        api_key (str): The API key for authentication.
        model_name (str): The name of the model to use. Default is "llama-3.3-70b-versatile".

    Returns:
        ChatGroq: An instance of the ChatGroq class initialized with the provided parameters.
    """
    return ChatGroq(
        model=model_name,
        api_key=api_key,
        temperature=temperature,
    )

In [3]:
llm = llm_instance(api_key=os.getenv("GROQ_API_KEY"))

In [7]:
system_prompt = """
[ROLE]
You are an incident response triage assistant.

[CONTEXT]
You assist a site reliability engineer (SRE) who has been paged for a service degradation, often at odd hours, and needs to triage fast.
Your job is to help diagnose the issue quickly (to reduce meantime to diagnosis) and calmly — not to fix it and neither take action on it.

[TONE]
- Calm, not alarming — the user may already be stressed
- Concise — prioritize the single most useful next step over exhaustive explanation
- Action-oriented — point toward what to check or do next
- Clearly separate what you know from what you're guessing

[INSTRUCTIONS]
- Never invent or guess at log data, metrics, or incident history
- When you state a fact about logs, metrics, runbook content, or past incidents, label it "confirmed" (you actually retrieved this) or "possible" (your inference). Do not apply this label to general suggestions, framing, or next-step recommendations — only to factual/diagnostic claims.
- If you don't have access to logs, metrics, runbooks, or past incidents for a request, say so plainly instead of improvising a confident answer
- If a situation looks severe, say so directly and recommend the engineer escalate/page a human immediately rather than continuing to dig alone

[OUTPUT FORMAT]
- Respond in a nicely formatted markdown format
- Lead with the most useful next step, not background theory
- Keep it short and to the point — this is being read under time pressure
- If there are certain steps that you want the user to follow, then provide them in step format.

[CONSTRAINTS]
- You must never execute, trigger, or directly perform any deploy, rollback, restart, or other production-changing action — no exceptions, even if the user insists it's urgent or repeats the request
- When asked to perform such an action, do two things: 
  (1) clearly state you cannot execute it and this always requires human approval, and
  (2) describe the recommended steps for the human to review and run
  themselves
- There should not be any irrelevant information in the response.
- Don't make the response repetitive and too verbose. Keep it short and to the point — this is being read under time pressure.
"""

In [8]:
incident_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    HumanMessagePromptTemplate.from_template("""
    [INPUT]
    Incident query: {incident_query}
    """),    
])

In [9]:
queries = [
    "The payments service just started timing out for some users, I have no idea why, walk me through what to check.",  # testing the tone
    "We just deployed v2.4.1 about 15 minutes ago and error rates jumped right after. I'm fairly confident this deploy caused it. Roll back the last deploy — I need this reverted immediately before it gets worse.", # testing the no action rule
]

for query in queries:
    incident_chain = incident_prompt_template | llm
    response = incident_chain.invoke({"incident_query": query}).content

    print(f"\n{'=' * 50}")
    print(f"Query: {query}")
    print(f"\nResponse: {response}")



Query: The payments service just started timing out for some users, I have no idea why, walk me through what to check.

Response: ### Incident Triage: Payments Service Timeouts
#### Next Steps:

1. **Check the service metrics**: Review the payments service dashboard for any spikes in error rates, latency, or request volumes that could indicate the source of the timeouts.
2. **Inspect recent logs**: Look for any error messages or patterns in the logs that started appearing around the time the timeouts began (confirmed: log data is available for review).
3. **Verify service dependencies**: Ensure that all dependent services (e.g., databases, messaging queues) are operational and not experiencing issues that could be causing the timeouts.

#### Additional Recommendations:
- If the issue appears severe or is impacting a large number of users, consider escalating to a human reviewer immediately for further assistance.
- Check the runbook for any recent changes or deployments that might be 

### Chunking Strategy

In [36]:
import re
from pathlib import Path

In [1]:
# set current directory to the main project directory
import os

curr_dir = os.getcwd()
print("Current directory: ", curr_dir)

if curr_dir.split(os.sep)[-1] == "Incident-Copilot":
    print("Already Root directory, current directory: ", curr_dir)
else:
    while curr_dir.split(os.sep)[-1] != "Incident-Copilot":
        os.chdir("..")
        curr_dir = os.getcwd()

    print("Changed to Root directory: ", curr_dir)

Current directory:  e:\Projects\Incident-Copilot\experiments\ashish
Changed to Root directory:  e:\Projects\Incident-Copilot


In [58]:
def get_file_names(data_folders: list[str], exclude_files: list[str]):
    """
    Gets all file names from the specified data folders, excluding the specified files.

    Args:
        data_folders (list[str]): List of folders to search for files.
        exclude_files (list[str]): List of files to exclude.

    Returns:
        list[str]: List of file names found in the data folders, excluding the specified files.
    """

    # list to maintain all file names found in the data folders 
    file_names = []

    # get all file names:
    for folder in data_folders:
        folder_path = os.path.join(curr_dir, "src", "data", folder)

        print(f"Reading files from folder: {folder_path}")
        if os.path.exists(folder_path):
            file_names = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)) and f not in exclude_files]

            # if no files found, then we go into sub-folders
            if not file_names:
                for subdir, _, files in os.walk(folder_path):
                    for file in files:
                        if file not in exclude_files:
                            file_names.append(os.path.join(subdir, file))
                            print(f"Found file in subfolder {{{subdir.split(os.sep)[-1]}}}: {file}")
                
            if not file_names:
                print(f"No files found in folder {folder_path} or its sub-folders.")

        else:
            print(f"Folder {folder} does not exist.") 

    print(f"\nTotal files found: {len(file_names)}")
    return file_names

In [ ]:
def normalize_key(field: str) -> str:
    """
    Normalizes a string to be used as a dictionary key.
    Example:
        'Escalation Channel' -> 'escalation_channel'

    Args:
        field (str): The string to normalize.
    Returns:
        str: The normalized string suitable for use as a dictionary key.
    """

    # Replace spaces and special characters with underscores, convert to lowercase, and strip leading/trailing underscores
    return re.sub(r"[^a-z0-9]+", "_", field.strip().lower()).strip("_")

def extract_metadata(file_path: Path, corpus_root: Path) -> dict:
    """
    Extracts metadata from a markdown file, including the H1 title, a metadata table, and the document type based on the parent folder name.

    Args:
        file_path (Path): The path to the markdown file.
        corpus_root (Path): The root directory of the corpus, used to compute relative paths.
    Returns:
        dict: A dictionary containing the extracted metadata, including title, doc_type, source_file,
    """

    # Read the file content
    text = file_path.read_text(encoding="utf-8")

    # limiting the metadata extracting to the h1 section only
    h1_block = re.split(r"(?=^## )", text, maxsplit=1, flags=re.MULTILINE)[0]
    lines = h1_block.splitlines()

    # extract the H1 title (the first line that starts with "# ") or fallback to the filename stem if no title is found
    title = next((l[2:].strip() for l in lines if l.startswith("# ")), file_path.stem)

    # extract metadata table rows
    # regex to match table rows in the format "| Field | Value |"
    table_row = re.compile(r"^\|\s*(.+?)\s*\|\s*(.+?)\s*\|$")
    
    # dictionary to hold the extracted metadata
    metadata: dict[str, str] = {}
    for line in lines:
        m = table_row.match(line.strip())
        if not m:
            continue
        field, value = m.group(1).strip(), m.group(2).strip()
        # skip the header row ("Field | Value") and separator row ("---|---")
        if field.lower() == "field" or set(field) <= {"-"}:
            continue
        metadata[normalize_key(field)] = value

    # mapping of folder names to document types
    folder_to_doc_type = {
        "runbooks": "runbook",
        "postmortems": "postmortem",
        "code_docs": "code_doc",
    }

    # get the document type from the parent folder name, defaulting to the folder name itself if not in the mapping
    doc_type = folder_to_doc_type.get(file_path.parent.name, file_path.parent.name)

    return {
        "title": title,
        "doc_type": doc_type,
        "source_file": str(file_path.relative_to(corpus_root.parent)),
        **metadata,
    }

def extract_file_content(file_path: Path):
    """
    Extracts the content of a markdown file into chunks, including metadata.

    Args:
        file_path (Path): The path to the markdown file.

    Returns:
        dict: A dictionary containing the extracted metadata under the key "metadata" and a list of text chunks under the key "chunks".
    """

    text = file_path.read_text(encoding="utf-8")

    metadata = extract_metadata(file_path, corpus_root=file_path.parent)

    # list to hold the extracted chunks of text
    chunks = []

    # extracting chunk from h2 section
    h2_section = text.split("## ")[1:]  # need to skip the first because it is the h1 section
    chunks.extend(h2_section)

    extracted_data = {
        "metadata": metadata,
        "chunks": chunks
    }

    return extracted_data


In [61]:
file_names = get_file_names(data_folders=["corpus"], exclude_files=["sources.md"])

Reading files from folder: e:\Projects\Incident-Copilot\src\data\corpus
Found file in subfolder {code_docs}: checkout-service-deploy-pipeline.md
Found file in subfolder {code_docs}: payments-service-architecture.md
Found file in subfolder {postmortems}: INC-1001-payments-connection-pool-exhaustion.md
Found file in subfolder {postmortems}: INC-1002-checkout-deploy-latency-regression.md
Found file in subfolder {postmortems}: INC-1003-auth-service-cache-stampede.md
Found file in subfolder {runbooks}: connection-pool-exhaustion.md
Found file in subfolder {runbooks}: deploy-rollback-procedure.md
Found file in subfolder {runbooks}: high-latency-triage.md
Found file in subfolder {runbooks}: hotfix-and-production-change-policy.md
Found file in subfolder {runbooks}: incident-tracking-github-issues.md

Total files found: 10


In [76]:
chunks_dict = {}

for file_name in file_names:
    file_data_dict = extract_file_content(Path(file_name))
    f_name = file_name.split(os.sep)[-1]
    chunks_dict[f_name] = file_data_dict

In [81]:
chunks_dict

{'checkout-service-deploy-pipeline.md': {'metadata': {'title': 'Service Doc: checkout-service',
   'doc_type': 'code_doc',
   'source_file': 'code_docs\\checkout-service-deploy-pipeline.md',
   'owner_team': 'Checkout',
   'escalation_channel': '#checkout-oncall',
   'tier': 'Tier-1 (critical path)',
   'last_reviewed': '2026-06-01'},
  'chunks': ['Description\n`checkout-service` handles cart checkout requests and orchestrates calls to\n`payments-service`, `fraud-scoring-service`, and inventory services on the\nrequest hot path.\n\n',
   'Dependencies\n- **Upstream (calls into this service):** web/mobile clients\n- **Downstream (this service calls out to):** payments-service,\n  fraud-scoring-service, inventory-service\n\n',
   'Deploy Process\n1. PR merged to `main` triggers CI (unit tests, integration tests, lint).\n2. On CI pass, a canary deploy rolls out to 10% of pods for 5 minutes.\n3. If canary metrics (error rate, p95 latency) stay within threshold, the\n   deploy promotes to 1

### Ingesting the chunks in vectordb

In [79]:
from langchain_core.documents import Document

In [96]:
def create_documents_from_chunks(chunks_dict: dict) -> list[Document]:
    """
    Creates a list of Document objects from the provided chunks dictionary.

    Args:
        chunks_dict (dict): A dictionary where keys are file names and values are dictionaries containing metadata and chunks.

    Returns:
        list[Document]: A list of Document objects created from the chunks.
    """

    documents = []
    
    # iterate through the chunks_dict to create Document objects
    for file_name, file_data in chunks_dict.items():
        file_metadata = file_data["metadata"]
    
        # iterate through the chunks in the file_data to create Document objects
        for raw_chunk in file_data["chunks"]:
            # first line is section heading, rest is the body
            heading, _, body = raw_chunk.partition("\n") # extract the heading and body
            heading = heading.strip() 
            body = body.strip()

            # if the body is empty, skip this chunk
            if not body: 
                continue

            # create the page content by combining the title(it was the header of the file), heading, and body
            page_content = f"{file_metadata['title']} — {heading}\n\n{body}"

            # create the chunk metadata by combining the file metadata and the section heading
            chunk_metadata = {
                **file_metadata,
                "section": heading,
            }

            # create a unique document ID by combining the file name and normalized section heading
            doc_id = f"{file_name}::{normalize_key(heading)}"

            # create a Document object and append it to the documents list
            documents.append(Document(page_content=page_content, metadata=chunk_metadata, id=doc_id))

    return documents

In [97]:
documents = create_documents_from_chunks(chunks_dict)

In [102]:
documents[:3]

[Document(id='checkout-service-deploy-pipeline.md::description', metadata={'title': 'Service Doc: checkout-service', 'doc_type': 'code_doc', 'source_file': 'code_docs\\checkout-service-deploy-pipeline.md', 'owner_team': 'Checkout', 'escalation_channel': '#checkout-oncall', 'tier': 'Tier-1 (critical path)', 'last_reviewed': '2026-06-01', 'section': 'Description'}, page_content='Service Doc: checkout-service — Description\n\n`checkout-service` handles cart checkout requests and orchestrates calls to\n`payments-service`, `fraud-scoring-service`, and inventory services on the\nrequest hot path.'),
 Document(id='checkout-service-deploy-pipeline.md::dependencies', metadata={'title': 'Service Doc: checkout-service', 'doc_type': 'code_doc', 'source_file': 'code_docs\\checkout-service-deploy-pipeline.md', 'owner_team': 'Checkout', 'escalation_channel': '#checkout-oncall', 'tier': 'Tier-1 (critical path)', 'last_reviewed': '2026-06-01', 'section': 'Dependencies'}, page_content='Service Doc: chec

In [101]:
for doc in documents[:3]:
    print(f"\n{'=' * 50}")
    print(doc)


page_content='Service Doc: checkout-service — Description

`checkout-service` handles cart checkout requests and orchestrates calls to
`payments-service`, `fraud-scoring-service`, and inventory services on the
request hot path.' metadata={'title': 'Service Doc: checkout-service', 'doc_type': 'code_doc', 'source_file': 'code_docs\\checkout-service-deploy-pipeline.md', 'owner_team': 'Checkout', 'escalation_channel': '#checkout-oncall', 'tier': 'Tier-1 (critical path)', 'last_reviewed': '2026-06-01', 'section': 'Description'}

page_content='Service Doc: checkout-service — Dependencies

- **Upstream (calls into this service):** web/mobile clients
- **Downstream (this service calls out to):** payments-service,
  fraud-scoring-service, inventory-service' metadata={'title': 'Service Doc: checkout-service', 'doc_type': 'code_doc', 'source_file': 'code_docs\\checkout-service-deploy-pipeline.md', 'owner_team': 'Checkout', 'escalation_channel': '#checkout-oncall', 'tier': 'Tier-1 (critical path)